# Phishing URL Detector
Trains a classifier on URL-based features to detect phishing sites.

**Pipeline:**
1. Load & merge two datasets
2. Extract 22 URL features via `feature_extract.py`
3. Train/test split → SMOTE → StandardScaler
4. Compare four models, optimising for **phishing recall**
5. Tune threshold via ROC + Youden's J
6. Save model to `phishing_model.pkl`


## 1. Imports

In [1]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_curve, auc
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from feature_extract import extract_features, FEATURE_NAMES

## 2. Load & Merge Datasets

In [2]:
df = pd.read_csv("phishing_site_urls.csv")
df = df.rename(columns={"URL": "url", "Label": "label"})
df["label"] = df["label"].map({"bad": 1, "good": 0})

df = (
    df.drop_duplicates(subset="url")
    .dropna(subset=["label"])
    .reset_index(drop=True)
)

df = df[df["url"].str.len() > 10]
df = df[df["url"].str.contains(r"\.", regex=True)]

print(f"Dataset shape: {df.shape}")
print(df["label"].value_counts())

## 3. Feature Extraction

Features are extracted via `feature_extract.py` — imported above.
This keeps training and inference in sync automatically.

In [3]:
feature_matrix = df["url"].apply(extract_features).tolist()
X = pd.DataFrame(feature_matrix, columns=FEATURE_NAMES)
y = df["label"]

print(X.shape)
X.head()

(516467, 22)


,suspicious_words,dots,hyphens,path_len,domain_digits,domain_len,domain_entropy,num_params,is_shortened,has_ip,...,subdomains,https,prefix_suffix,redirect,abnormal_www,double_slash,special_chars,suspicious_tld,brand,free_ddns
0,0,3,0,11,0,19,3.195296,0,0,0,...,2,0,0,0,0,0,3,0,0,0
1,0,1,0,47,0,23,3.708132,0,0,0,...,1,0,0,0,0,0,1,0,0,0
2,1,4,1,20,0,50,3.999080,3,0,0,...,4,1,1,0,0,0,9,0,1,0
3,0,2,0,0,0,11,3.095795,0,0,0,...,2,0,0,0,0,0,2,0,0,0
4,0,2,2,33,0,15,3.189898,0,0,0,...,2,0,0,0,0,0,4,0,0,0


## 4. Train / Test Split → SMOTE → Scaling

**Important ordering:** split first, then SMOTE only on the training set.
Doing SMOTE before splitting leaks synthetic samples into the test set.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Oversample minority class in training data only
X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)

# Scale — fit on training data only, apply to both
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class balance: {dict(zip(*np.unique(y_train, return_counts=True)))}")

Train: (634846, 22), Test: (103294, 22)
Train class balance: {0: 317423, 1: 317423}


## 5. Model Comparison

Four models compared. Threshold is tuned to maximise **phishing F1 (class 1)** — 
not overall F1. For a security tool, missing phishing is more dangerous than a false alarm.
XGBoost uses `scale_pos_weight=5` to tell the model phishing errors cost 5x more.


In [5]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {ratio:.2f}")

print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=600, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=ratio,
    min_child_weight=3, gamma=0.05, reg_alpha=0.1, reg_lambda=1.5,
    random_state=42, eval_metric="logloss", n_jobs=-1,
)
xgb.fit(X_train, y_train)

print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=300, max_depth=25, min_samples_split=5,
    min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1,
)
rf.fit(X_train, y_train)

# Soft voting ensemble
prob_xgb = xgb.predict_proba(X_test)[:, 1]
prob_rf  = rf.predict_proba(X_test)[:, 1]
y_prob   = (prob_xgb + prob_rf) / 2


## 6. Final Model — Threshold Tuning via ROC / Youden's J

Youden's J (TPR − FPR) gives a principled threshold that balances
sensitivity and specificity, rather than just maximising F1 on one split.

In [6]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Optimise threshold for macro F1 (balanced between legit and phishing)
thresholds = roc_thresholds
scores = [f1_score(y_test, (y_prob > t).astype(int), average="macro") for t in thresholds]
best_idx       = int(np.argmax(scores))
best_threshold = thresholds[best_idx]

print(f"Best threshold (macro F1): {best_threshold:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, lw=2, label=f"Ensemble (AUC = {roc_auc:.2f})")
plt.scatter(fpr[best_idx], tpr[best_idx], marker="o", color="red",
            label=f"Best threshold = {best_threshold:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Ensemble (XGBoost + Random Forest)")
plt.legend()
plt.tight_layout()
plt.show()

In [7]:
y_pred = (y_prob > best_threshold).astype(int)
print(f"Final accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

## 7. Feature Importance

In [8]:
# Average feature importance across both models
importance = pd.Series(
    (xgb.feature_importances_ + rf.feature_importances_) / 2,
    index=FEATURE_NAMES
).sort_values(ascending=False)

importance.plot(kind="bar", figsize=(12, 4), title="Feature Importances (Ensemble Average)")
plt.tight_layout()
plt.show()
print(importance)

## 8. Save Model

In [10]:
model_data = {
    "model":     (xgb, rf),   # ensemble: (XGBoost, RandomForest)
    "scaler":    scaler,
    "threshold": best_threshold,
    "features":  FEATURE_NAMES,
}

with open("phishing_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Saved -> phishing_model.pkl")
print(f"Threshold: {best_threshold:.4f}")

In [11]:
import pickle
with open("phishing_model.pkl", "rb") as f:
    saved = pickle.load(f)

print("Features model was trained on:", len(saved["features"]))
print(saved["features"])

Features model was trained on: 22
['suspicious_words', 'dots', 'hyphens', 'path_len', 'domain_digits', 'domain_len', 'domain_entropy', 'num_params', 'is_shortened', 'has_ip', 'has_at', 'url_length', 'subdomains', 'https', 'prefix_suffix', 'redirect', 'abnormal_www', 'double_slash', 'special_chars', 'suspicious_tld', 'brand', 'free_ddns']


In [12]:
import sys
sys.path.insert(0, '.')
from feature_extract import FEATURE_NAMES
print(len(FEATURE_NAMES))
print(FEATURE_NAMES)

22
['suspicious_words', 'dots', 'hyphens', 'path_len', 'domain_digits', 'domain_len', 'domain_entropy', 'num_params', 'is_shortened', 'has_ip', 'has_at', 'url_length', 'subdomains', 'https', 'prefix_suffix', 'redirect', 'abnormal_www', 'double_slash', 'special_chars', 'suspicious_tld', 'brand', 'free_ddns']
